# The small run, cell by cell

The deterministic small example (d=3, 12 rounds, zero noise, 3 sliding
windows) run end to end, showing every input and every output:

1. the yaml that names every cost,
2. the Stim circuit the QPU replays,
3. the run itself,
4. what the QPU put out (detection events and the truth),
5. every round's journey to Buffer 0,
6. every window's timestamps,
7. the same timestamps by hand, checked cell for cell.

This notebook is a viewing layer only: the experiment runs without it
(`analytic_small_run.py` and `simulated_small_run.py` in this folder do the
same from the shell). See `README.md` here for the full walkthrough.

In [1]:
# Make the repository root and this folder importable, wherever jupyter started.
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "guide" / "walkthrough"))
sys.path.insert(0, str(repo_root))
print(repo_root)

/scratch/gpfs/MARTONOSI/sk2415/qlx-qec-sandbox/decsim


## 1. The input: one yaml names every cost

Round period 1.0 µs, QC 0.15, C2B 0.10, CWD 2.0, DD 0.5, WDO 1.0,
algorithm 0.028, engine 250 MHz, frame commit 0.004. Nothing else costs
time, so the whole run is checkable by hand.

In [2]:
from experiments.baseline.baseline_closed_loop import load_config

CONFIG_PATH = repo_root / "experiments/validation/analytic_oracle_d3_r12.yaml"
print(CONFIG_PATH.read_text())

# Deterministic analytical-oracle case.
#
# Purpose:
#   Verify the exact baseline event ordering and timing, not performance or LER.
#   This case intentionally has:
#     - zero physical noise,
#     - one fixed decoder latency,
#     - one decoder unit,
#     - unbounded link bandwidth,
#     - only 12 QEC rounds, which produce exactly 3 sliding windows.
#
# Expected analytical results are provided beside this file.

code_task: surface_code:rotated_memory_z
distance: 3
rounds_per_shot: 12

windowing:
  scheme: sliding
  commit_rounds: 3
  buffer_rounds: 3

sweep:
  # p = 0 exactly cannot run: a noiseless circuit has an empty detector error
  # model and no matching graph. 1.0e-9 samples zero defects on every shot and
  # every timing tick is identical to p = 0 (preset stage costs, payload sizes
  # independent of p).
  - physical_error_probability: [1.0e-9]
    round_period_us: [1.0]
    algorithm_latency_us: [0.028]
    shots: 1

controller:
  t_binary_availability_us: 0.0
  t_pack

## 2. The input: the Stim circuit the QPU replays

A d=3 rotated surface code memory: 9 data qubits, 8 ancillas, 12 rounds,
one final data measurement. The detector counts per round are exactly the
syndrome payload bits the links serialize.

In [3]:
from experiments.baseline.baseline_closed_loop import memory_circuit

config = load_config(CONFIG_PATH)
physical_error_probability = config["sweep"][0]["physical_error_probability"][0]
circuit = memory_circuit(config, physical_error_probability)
print(circuit)

QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
R 1 3 5 8 10 12 15 17 19
X_ERROR(1e-09) 1 3 5 8 10 12 15 17 19
R 2 9 11 13 14 16 18 25
X_ERROR(1e-09) 2 9 11 13 14 16 18 25
TICK
DEPOLARIZE1(1e-09) 1 3 5 8 10 12 15 17 19
H 2 11 16 25
DEPOLARIZE1(1e-09) 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
DEPOLARIZE2(1e-09) 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
DEPOLARIZE2(1e-09) 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
DEPOLARIZE2(1e-09) 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
DEPOLARIZE2(1e-09) 16 8 11 3 25 17 1 9 10 18 5 13
TICK
H 2 11 16 25
DEPOLARIZE1(1e-09) 2 11 16 25
TICK
X

In [4]:
from collections import Counter

detectors_per_round = Counter(int(coordinates[2])
                              for coordinates in circuit.get_detector_coordinates().values())
print(f"detectors: {circuit.num_detectors}, logical observables: {circuit.num_observables}")
print("detectors per round (0-based):", dict(sorted(detectors_per_round.items())))

detectors: 96, logical observables: 1
detectors per round (0-based): {0: 4, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8, 7: 8, 8: 8, 9: 8, 10: 8, 11: 8, 12: 4}


## 3. Run it

`build_run` assembles the whole reaction path from the yaml; `spec.build()`
replays the shot through the event loop.

In [5]:
from experiments.baseline.baseline_closed_loop import build_run

block = config["sweep"][0]
spec, decoder_engine = build_run(
    config,
    physical_error_probability=block["physical_error_probability"][0],
    round_period_us=block["round_period_us"][0],
    algorithm_latency_us=block["algorithm_latency_us"][0],
    seed=0)
completed = spec.build()
print(completed.result.terminal_status)

complete


## 4. What the QPU put out

At p = 1e-9 every shot samples zero detection events, so the truth and the
loop's prediction are both "no flip". The bits below are the actual decoder
input, not a summary.

In [6]:
operation = completed.result.operation_results[0]
events = completed.qpu.model.sampled_detection_events(operation.operation_id)
print(f"detection events sampled: {len(events)}, fired: {sum(events)}")
print(f"observable truth: {operation.observable_truth}")
print(f"loop prediction:  {operation.logical_observables}")

detection events sampled: 96, fired: 0
observable truth: (0,)
loop prediction:  (0,)


## 5. Every round's journey to Buffer 0

Round r leaves the QPU at r x 1.0 µs and is published a constant 0.25 µs
later (QC 0.15 + C2B 0.10; controller processing and packing are zero
here). Same frequency in and out, only the phase shifts. The payload is the
round's detector count: 4 bits for round 1, 8 for the middle rounds, 12 for
round 12 (the final layer folds into it).

In [7]:
from decsim.config import TICKS_PER_US


def us(ticks):
    return ticks / TICKS_PER_US


transfers = completed.result.link_traffic["transfers"]
qc_send = {t["attribution"]["round_lo"]: t["send_ticks"]
           for t in transfers if t["path"] == "qc"}
publication = {t["attribution"]["round_lo"]: t["delivery_ticks"]
               for t in transfers if t["path"] == "c2b"}
payload_bits = {t["attribution"]["round_lo"]: t["payload_bits"]
                for t in transfers if t["path"] == "c2b"}

print("round | payload bits | leaves QPU (µs) | published in Buffer 0 (µs) | offset (µs)")
for round_index in sorted(qc_send):
    send_us = us(qc_send[round_index])
    published_us = us(publication[round_index])
    print(f"{round_index:5} | {payload_bits[round_index]:12} | {send_us:15.3f} "
          f"| {published_us:26.3f} | {published_us - send_us:11.3f}")

round | payload bits | leaves QPU (µs) | published in Buffer 0 (µs) | offset (µs)
    1 |            4 |           1.000 |                      1.250 |       0.250
    2 |            8 |           2.000 |                      2.250 |       0.250
    3 |            8 |           3.000 |                      3.250 |       0.250
    4 |            8 |           4.000 |                      4.250 |       0.250
    5 |            8 |           5.000 |                      5.250 |       0.250
    6 |            8 |           6.000 |                      6.250 |       0.250
    7 |            8 |           7.000 |                      7.250 |       0.250
    8 |            8 |           8.000 |                      8.250 |       0.250
    9 |            8 |           9.000 |                      9.250 |       0.250
   10 |            8 |          10.000 |                     10.250 |       0.250
   11 |            8 |          11.000 |                     11.250 |       0.250
   12 |         

## 6. The output: three windows, timestamp by timestamp

Every timestamp read out of the simulator's own records (window manager,
link transfer log, Pauli frame).

In [8]:
from analytic_small_run import COLUMNS, analytic_timeline, print_table
from simulated_small_run import differences, simulated_timeline

simulated_rows = simulated_timeline(config)
print_table(simulated_rows)

| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |


## 7. The same table by hand

Five formulas produce the run (see README section 1). The check compares
every cell of the hand table against the simulator's table.

In [9]:
analytic_rows = analytic_timeline(config)
print_table(analytic_rows)

disagreements = differences(analytic_rows, simulated_rows)
assert not disagreements, disagreements
print(f"\nMATCH: all {len(analytic_rows) * len(COLUMNS)} cells agree to the tick.")

| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |

MATCH: all 36 cells agree to the tick.


## Try a knob

Change one number in the yaml (say `cwd.latency_us` to 3.0, or
`round_period_us` to 0.5), rerun all cells, and redo the arithmetic of
README section 1: the tables must still agree to the tick. The frequency
sweep and its figures are README section 5.